In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

BASE_DIR = Path.cwd().parent
processed_dir = BASE_DIR / "data" / "processed"
hexagons_gdf = gpd.read_file(processed_dir / 'hexagons_final.geojson')

pvz_df = pd.read_csv(processed_dir / 'pvz_krasnoyarsk.csv')
geometry = [Point(xy) for xy in zip(pvz_df.longitude, pvz_df.latitude)]
pvz_gdf = gpd.GeoDataFrame(pvz_df, geometry=geometry, crs="EPSG:4326")


buildings_gdf = gpd.read_file(processed_dir / 'buildings.geojson')

# 4. Перевод всех слоев в проекционную систему координат (EPSG:32646 для Красноярска)
hexagons_gdf = hexagons_gdf.to_crs(epsg=32646)
pvz_gdf = pvz_gdf.to_crs(epsg=32646)
buildings_gdf = buildings_gdf.to_crs(epsg=32646)

print(f"Загружено гексагонов: {len(hexagons_gdf)}")
print(f"Загружено ПВЗ: {len(pvz_gdf)}")
print(f"Загружено зданий: {len(buildings_gdf)}")

<class 'pathlib._local.WindowsPath'>
Загружено гексагонов: 483
Загружено ПВЗ: 420
Загружено зданий: 25167


In [ ]:
pvz_in_hex = gpd.sjoin(pvz_gdf, hexagons_gdf, how="inner", predicate="intersects")

# Считаем количество ПВЗ по ID гексагона
hex_id_col = 'h3_id' 

pvz_counts = pvz_in_hex.groupby(hex_id_col).size().reset_index(name='pvz_count')

# Добавляем данные о количестве ПВЗ к исходному гео-датафрейму гексагонов
hexagons_with_pvz = hexagons_gdf.merge(pvz_counts, on=hex_id_col, how='left')

hexagons_with_pvz['pvz_count'] = hexagons_with_pvz['pvz_count'].fillna(0).astype(int)

print(f"Максимальное число ПВЗ в одном гексагоне: {hexagons_with_pvz['pvz_count'].max()}")
print(f"Гексагонов с хотя бы одним ПВЗ: {(hexagons_with_pvz['pvz_count'] > 0).sum()}")

Максимальное число ПВЗ в одном гексагоне: 8
Гексагонов с хотя бы одним ПВЗ: 145


In [ ]:
hexagons_with_pvz['population_estimated'] = hexagons_with_pvz['population_estimated'].fillna(0).astype(int)

# Рассчитываем Индекс Доступности: количество ПВЗ на 10 000 человек
# Если в гексагоне 0 населения, индекс тоже считаем равным 0, чтобы избежать ошибки деления на ноль.
def calculate_accessibility(row):
    if row['population_estimated'] > 0:
        return (row['pvz_count'] / row['population_estimated']) * 10000
    else:
        return 0.0

hexagons_with_pvz['pvz_per_10k_people'] = hexagons_with_pvz.apply(calculate_accessibility, axis=1)

# Смотрим результаты: Топ-5 гексагонов с лучшей доступностью
# (Отфильтруем гексагоны с крошечным населением < 50 человек, чтобы избежать выбросов вида "1 ПВЗ на 2 человека")
top_accessible = hexagons_with_pvz[hexagons_with_pvz['population_estimated'] >= 50].sort_values(by='pvz_per_10k_people', ascending=False)

print("\nТоп-5 гексагонов с самой ВЫСОКОЙ доступностью ПВЗ (на 10 000 чел.):")
# Выводим ID, кол-во ПВЗ, население и сам индекс
print(top_accessible[['h3_id', 'pvz_count', 'population_estimated', 'pvz_per_10k_people']].head())

# 4. Анализ потенциала: гексагоны с большим населением, но БЕЗ ПВЗ
no_pvz_high_pop = hexagons_with_pvz[hexagons_with_pvz['pvz_count'] == 0].sort_values(by='population_estimated', ascending=False)

print("\nТоп-5 гексагонов с наибольшим населением, но БЕЗ ПВЗ (Потенциал для открытия):")
print(no_pvz_high_pop[['h3_id', 'pvz_count', 'population_estimated']].head())


Топ-5 гексагонов с самой ВЫСОКОЙ доступностью ПВЗ (на 10 000 чел.):
               h3_id  pvz_count  population_estimated  pvz_per_10k_people
27   880b9a2b65fffff          1                   183           54.644809
284  880b9a7605fffff          2                   471           42.462845
322  880b9a75c7fffff          2                   902           22.172949
295  880b9a6641fffff          1                   552           18.115942
171  880b9a7635fffff          3                  1665           18.018018

Топ-5 гексагонов с наибольшим населением, но БЕЗ ПВЗ (Потенциал для открытия):
               h3_id  pvz_count  population_estimated
482  880b9a296bfffff          0                 17086
32   880b9a7657fffff          0                 10554
73   880b9a7583fffff          0                  9553
131  880b9a74c1fffff          0                  7937
31   880b9a7459fffff          0                  7659


In [8]:
import os

# Сохраняем итоговый гео-датафрейм в новый файл
output_path = os.path.join(processed_dir / 'hexagons_analyzed.geojson')
hexagons_with_pvz.to_file(output_path, driver='GeoJSON')
print(f"Данные успешно сохранены в: {output_path}")

Данные успешно сохранены в: c:\Users\user\Documents\0study\projects\Optimization-of-Ozon-pick-up-points-in-Krasnoyarsk\data\processed\hexagons_analyzed.geojson


In [18]:
import os
import json
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MeasureControl
from shapely.geometry import shape, Point

HEX_PATH = os.path.join(processed_dir / 'hexagons_final.geojson')
PVZ_PATH = os.path.join(processed_dir / 'pvz_krasnoyarsk.csv')


def load_geojson(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    features = data['features']
    geoms = [shape(f['geometry']) for f in features]
    props = [f['properties'] for f in features]
    return gpd.GeoDataFrame(props, geometry=geoms, crs="EPSG:4326")

hexagons = load_geojson(HEX_PATH)
pvz_df = pd.read_csv(PVZ_PATH)
pvz_gdf = gpd.GeoDataFrame(
    pvz_df, 
    geometry=[Point(xy) for xy in zip(pvz_df.longitude, pvz_df.latitude)], 
    crs="EPSG:4326"
)

# Анализ (считаем ПВЗ в гексагонах)
# Работаем в 4326 для Folium, sjoin справится
pvz_in_hex = gpd.sjoin(pvz_gdf, hexagons, how="inner", predicate="intersects")
pvz_counts = pvz_in_hex.groupby('h3_id').size().reset_index(name='pvz_count')
hex_results = hexagons.merge(pvz_counts, on='h3_id', how='left').fillna({'pvz_count': 0})

# Расчет индекса и потенциала
def calc_metrics(row):
    pop = row.get('population_estimated', 0)
    index = (row['pvz_count'] / pop * 10000) if pop > 0 else 0
    blind_pop = pop if row['pvz_count'] == 0 else 0
    return pd.Series([index, blind_pop])

hex_results[['pvz_index', 'blind_spot_pop']] = hex_results.apply(calc_metrics, axis=1)

# Инициализация карты (центрируемся на Красноярске)
m = folium.Map(location=[56.01, 92.85], zoom_start=12, tiles='cartodbpositron')

# Слой 1: Индекс доступности (Choropleth)
cp_index = folium.Choropleth(
    geo_data=hex_results,
    name='Индекс доступности ПВЗ',
    data=hex_results,
    columns=['h3_id', 'pvz_index'],
    key_on='feature.properties.h3_id',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='ПВЗ на 10 000 человек',
    highlight=True
).add_to(m)

# Добавляем подсказки при наведении для индекса
folium.GeoJsonTooltip(['h3_id', 'population_estimated', 'pvz_count', 'pvz_index']).add_to(cp_index.geojson)

# Слой 2: Слепые зоны (где высокий потенциал)
cp_blind = folium.Choropleth(
    geo_data=hex_results[hex_results['blind_spot_pop'] > 0],
    name='Слепые зоны (Потенциал)',
    data=hex_results,
    columns=['h3_id', 'blind_spot_pop'],
    key_on='feature.properties.h3_id',
    fill_color='Reds',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Население без доступа к ПВЗ',
    show=False # Скрыт по умолчанию
).add_to(m)

# Слой 3: Точки существующих ПВЗ
pvz_layer = folium.FeatureGroup(name="Существующие ПВЗ")
for _, row in pvz_df.iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=3,
        color='blue',
        fill=True,
        popup=f"Адрес: {row.address_clean}",
    ).add_to(pvz_layer)
pvz_layer.add_to(m)

folium.LayerControl().add_to(m)
m.add_child(MeasureControl())

output_html = BASE_DIR / 'reports' / 'krasnoyarsk_interactive_map.html'
output_html.parent.mkdir(parents=True, exist_ok=True)
m.save(output_html)

print(f"Интерактивная карта создана: {output_html}")

Интерактивная карта создана: c:\Users\user\Documents\0study\projects\Optimization-of-Ozon-pick-up-points-in-Krasnoyarsk\reports\krasnoyarsk_interactive_map.html
